In [2]:
import json
import spacy
import random


random.seed(42)

print("Loading spaCy model...")
nlp = spacy.load("en_core_web_sm")

def create_mixed_datasets(input_file="ebm_nlp_ground_truth.json"):
    print(f"Loading ground truth data from {input_file}...")
    with open(input_file, "r", encoding="utf-8") as f:
        dataset = json.load(f)
        
    
    train_data = [d for d in dataset if d.get('split') == 'train']
    test_data = [d for d in dataset if d.get('split') == 'test']
    
    def process_split(data, split_name, mask_ratio=0.50):
        
        valid_data = [d for d in data if any(d['spans'].values())]
        random.shuffle(valid_data)
        
        target_masked_count = int(len(valid_data) * mask_ratio)
        masked_count = 0
        
        mixed_examples = []
        
        for abstract in valid_data:
            
            if masked_count < target_masked_count:
                available_slots = [slot for slot, spans in abstract['spans'].items() if spans]
                target_slot = random.choice(available_slots)
                spans_to_remove = abstract['spans'][target_slot]
                
                doc = nlp(abstract['text'])
                sentences = [sent.text.strip() for sent in doc.sents]
                
                masked_abstract = []
                withheld_sentences = []
                
                for sentence in sentences:
                    contains_span = any(span.lower() in sentence.lower() for span in spans_to_remove)
                    if contains_span:
                        withheld_sentences.append(sentence)
                    else:
                        masked_abstract.append(sentence)
                        
                
                if withheld_sentences:
                    mixed_examples.append({
                        "pmid": abstract['pmid'],
                        "split": split_name,
                        "is_masked": True,
                        "missing_slot": target_slot,
                        "model_input_text": " ".join(masked_abstract),
                        "original_text": abstract['text'],
                        "withheld_text": " ".join(withheld_sentences),
                        "target_spans": spans_to_remove
                    })
                    masked_count += 1
                    continue 
                    
            
            mixed_examples.append({
                "pmid": abstract['pmid'],
                "split": split_name,
                "is_masked": False,
                "missing_slot": None,
                "model_input_text": abstract['text'],
                "original_text": abstract['text'],
                "withheld_text": None,
                "target_spans": None
            })
            
        
        random.shuffle(mixed_examples)
        
       
        actual_masked = sum(1 for d in mixed_examples if d['is_masked'])
        print(f"  -> Generated {actual_masked} masked and {len(mixed_examples) - actual_masked} complete abstracts.")
        
        return mixed_examples

    print(f"\nProcessing Train Split ({len(train_data)} total abstracts)...")
    train_mixed = process_split(train_data, "train")
    
    print(f"\nProcessing Test Split ({len(test_data)} total abstracts)...")
    test_mixed = process_split(test_data, "test")
    
    
    train_file = "ebm_nlp_train_mixed.json"
    with open(train_file, "w", encoding="utf-8") as f:
        json.dump(train_mixed, f, indent=2)
    print(f"\nSuccessfully saved {len(train_mixed)} train examples to {train_file}")
    
   
    test_file = "ebm_nlp_test_mixed.json"
    with open(test_file, "w", encoding="utf-8") as f:
        json.dump(test_mixed, f, indent=2)
    print(f"Successfully saved {len(test_mixed)} test examples to {test_file}")


create_mixed_datasets()

Loading spaCy model...
Loading ground truth data from ebm_nlp_ground_truth.json...

Processing Train Split (4792 total abstracts)...
  -> Generated 2396 masked and 2396 complete abstracts.

Processing Test Split (189 total abstracts)...
  -> Generated 94 masked and 95 complete abstracts.

Successfully saved 4792 train examples to ebm_nlp_train_mixed.json
Successfully saved 189 test examples to ebm_nlp_test_mixed.json
